In [ ]:
"""
For Google Colab.
"""

# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo
%cd /content
!git clone https://github.com/AHHHHHH0-0/Extending-CoFinDiff.git
%cd Extending-CoFinDiff

# Open terminal and swith branch 

# Change file paths 

## Import


In [ ]:
import sys
import json
import torch
import wandb
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, str(Path().resolve().parent))
from denoiser.unet_model import UNetDenoiser
from preprocessing.condition_encoder import ConditionEncoder
from diffusion.diffusion import Diffusion
from training import train_step, validate, FinancialDataset
from config import training_config, project_config

## Config


In [ ]:
# Set seed 
SEED = project_config.SEED
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Device
DEVICE = project_config.DEVICE

# Hyperparameters
BS = training_config.BATCH_SIZE
LR = training_config.LEARNING_RATE
EP = training_config.EPOCHS
WD = training_config.WEIGHT_DECAY
PU = training_config.P_UNCOND
ES = training_config.EARLY_STOPPING

print(f"Device: {DEVICE}")
print(f"Hyperparameters:")
print(f"  Batch size: {BS}")
print(f"  Learning rate: {LR}")
print(f"  Max epochs: {EP}")
print(f"  Weight decay: {WD}")
print(f"  P(uncond): {PU}")
print(f"  Early stopping: {ES}")


## Load data


In [ ]:
# Load dataset
dataset = FinancialDataset('../data/train/train_data_2d.json')

# Inspect a sample
sample = dataset[0]
print(f"\nSample structure:")
for key, val in sample.items():
    print(f"  {key}: shape {val.shape}, dtype {val.dtype}")


In [ ]:
# 80-20 train-val split
train_indices, val_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

print(f"Train-Validation Split Setup:")
print(f"  Total samples: {len(dataset)}")
print(f"  Train samples: {len(train_indices)} ({len(train_indices)/len(dataset)*100:.1f}%)")
print(f"  Validation samples: {len(val_indices)} ({len(val_indices)/len(dataset)*100:.1f}%)")

## Training set-up


In [ ]:
# Create checkpoints directory
checkpoint_dir = Path('../../models/checkpoints')
checkpoint_dir.mkdir(exist_ok=True, parents=True)

# Storage for training results
training_results = {
    'train_losses': [],
    'val_losses': [],
    'best_val_loss': float('inf'),
    'final_epoch': 0
}

print(f"Set up checkpoint directory!")

In [ ]:
# Data subsets
train_subset = dataset.get_subset(train_indices)
val_subset = dataset.get_subset(val_indices)

# Data loaders
train_loader = DataLoader(
    train_subset,
    batch_size=BS,
    shuffle=True,
    num_workers=0,  # Use 0 for notebook, can increase for scripts
    pin_memory=True if torch.cuda.is_available() else False,
    generator=torch.Generator().manual_seed(SEED)  # Seeded generator
)
val_loader = DataLoader(
    val_subset,
    batch_size=BS,
    shuffle=False,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

# Models
denoiser = UNetDenoiser(in_channels=1).to(DEVICE)
cond_encoder = ConditionEncoder().to(DEVICE)
diffusion = Diffusion()

# Optimizer
optimizer = torch.optim.Adam(
    list(denoiser.parameters()) + list(cond_encoder.parameters()),
    lr=LR,
    weight_decay=WD
)

# LR Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EP)

# Training tracking
train_losses = []
val_losses = []
best_val_loss = float('inf')
epochs_without_improvement = 0

print(f"Set up models and metrics!")

In [ ]:
run = wandb.init(
    entity="ahliu7-uci",
    project="Extending-CoFinDiff", 
    name="testing_1",
    dir="..",  # Save wandb folder in root directory
    config={
        "batch_size": BS,
        "learning_rate": LR,
        "epochs": EP,
        "weight_decay": WD,
        "p_uncond": PU,
        "early_stopping": ES
    },
)

print(f"Set up wandb!")

In [ ]:
# Training Loop
for epoch in range(EP):
    # Train mode
    denoiser.train()
    cond_encoder.train()
    
    # Keep track of loss and batches
    epoch_train_loss = 0.0
    num_train_batches = 0
    
    # Progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EP}", leave=False)

    # Iterate over batches
    for batch in pbar:
        # Get data and add channel dimension
        x0 = batch['returns_2d'].unsqueeze(1).to(DEVICE)  # (B, 1, H, W)
        
        # Prepare conditions
        conditions = {
            'trend': batch['trend'].to(DEVICE),
            'realized_vol': batch['realized_vol'].to(DEVICE),
            'interest_rate': batch['interest_rate'].to(DEVICE),
            'volatility_index': batch['volatility_index'].to(DEVICE)
        }
        
        # Training step
        loss = train_step(
            denoiser=denoiser,
            diffusion=diffusion,
            x=x0,
            conditions=conditions,
            cond_encoder=cond_encoder,
            optimizer=optimizer,
            p_uncond=PU
        )
        
        epoch_train_loss += loss
        num_train_batches += 1
        
        pbar.set_postfix({"loss": f"{loss:.6f}"})
    
    # Train loss per epoch
    avg_train_loss = epoch_train_loss / num_train_batches
    train_losses.append(avg_train_loss)
    
    # Validation loss per epoch
    avg_val_loss = validate(denoiser, cond_encoder, diffusion, val_loader, DEVICE)
    val_losses.append(avg_val_loss)

    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss, 
        "val_loss": avg_val_loss,
        "lr": scheduler.get_last_lr()[0]
        })
    
    # Update learning rate
    scheduler.step()
    
    # Print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{EP} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
    
    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        
        # Save best model
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': denoiser.state_dict(),
            'cond_encoder_state_dict': cond_encoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'best_val_loss': best_val_loss
        }
        torch.save(checkpoint, checkpoint_dir / 'best_model.pt')
    else:
        epochs_without_improvement += 1
    
    # Early stopping
    if epochs_without_improvement >= ES:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}")
        print(f"Best validation loss: {best_val_loss:.6f}")
        break

# Finish wandb run
run.finish()

# Store training results
training_results = {
    'train_losses': train_losses,
    'val_losses': val_losses,
    'best_val_loss': best_val_loss,
    'final_epoch': len(train_losses)
}

print(f"\n{'='*80}")
print("TRAINING COMPLETE!")
print(f"{'='*80}")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Stopped at epoch: {len(train_losses)}")

# Section E: Results Tracking and Visualization


In [ ]:
# Training Results Summary
print("\nTraining Results Summary:")
print(f"{'='*60}")
print(f"Best validation loss: {training_results['best_val_loss']:.6f}")
print(f"Final train loss: {training_results['train_losses'][-1]:.6f}")
print(f"Final validation loss: {training_results['val_losses'][-1]:.6f}")
print(f"Training stopped at epoch: {training_results['final_epoch']}")
print(f"{'='*60}")

In [ ]:
# Save results summary to JSON
results_summary = {
    'train_test_split': {
        'train_size': len(train_indices),
        'val_size': len(val_indices),
        'split_ratio': '80-20'
    },
    'final_results': {
        'best_val_loss': float(training_results['best_val_loss']),
        'final_epoch': int(training_results['final_epoch']),
        'final_train_loss': float(training_results['train_losses'][-1]),
        'final_val_loss': float(training_results['val_losses'][-1])
    },
    'training_history': {
        'train_losses': [float(x) for x in training_results['train_losses']],
        'val_losses': [float(x) for x in training_results['val_losses']]
    }
}

# Save to JSON
with open(checkpoint_dir / 'training_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"Results summary saved to {checkpoint_dir / 'training_results.json'}")
print("\nTraining complete! Best model is saved as 'best_model.pt'")